In [ ]:
# Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import warnings


# Import our custom data access client
# Restart imports to pick up new methods
import importlib
import src.aqf.data_access
import src.aqf.tickers
import src.aqf.dcc_lightning
importlib.reload(src.aqf.data_access)
importlib.reload(src.aqf.tickers)
importlib.reload(src.aqf.dcc_lightning)
from src.aqf.data_access import FirstRateDataClient
from src.aqf.tickers_full import NASDAQ_TICKERS

# Configure display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
warnings.filterwarnings('ignore')

# Configure matplotlib
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 8)

# Configure plotly
import plotly.io as pio
pio.renderers.default = 'notebook'

In [ ]:
# Initialize the FirstRateData client
client = FirstRateDataClient(profile_name="firstratedata")

# Get overview of available data
print("\nData Availability Overview:")
dates_info = client.get_available_dates()
for key, value in dates_info.items():
    print(f"   {key}: {value}")

In [ ]:
start_date = '2021-07-01'
end_date = '2021-09-30' 

print(f"\nLoading {len(NASDAQ_TICKERS)} tickers from {start_date} to {end_date} ...")

try:
    multi_ticker_data = client.load_multi_ticker_data(
        tickers=NASDAQ_TICKERS,
        start_date=start_date,
        end_date=end_date,
        frequency='1T',  
        add_features=True,
        fill_method='drop'
    )
    
    print(f"Successfully loaded data!")
    
    # Display sample data
    print(f"\nSample data (first 5 rows):")
    print(multi_ticker_data.head())
    
except Exception as e:
    print(f"Error loading multi-ticker data: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
if 'multi_ticker_data' in locals() and not multi_ticker_data.empty:
    try:
        # Create aligned dataset (tickers as columns)
        aligned_data = client.create_aligned_dataset(
            multi_ticker_data, 
            value_column='close',  # Use close prices
            fill_method='forward'
        )
        
        print(f"Successfully created aligned dataset!")
        print(f"   Missing values per ticker:")
        for col in aligned_data.columns:
            missing = aligned_data[col].isna().sum()
            print(f"      {col}: {missing} ({missing/len(aligned_data)*100:.1f}%)")
        
        # Display sample aligned data
        print(f"\nSample aligned data (first 5 rows):")
        print(aligned_data.head())
        
        # Plot aligned data
        fig, ax = plt.subplots(figsize=(15, 8))
        for ticker in aligned_data.columns:
            ax.plot(aligned_data.index, aligned_data[ticker], label=ticker, alpha=0.8)
        
        ax.set_title('Aligned Stock Prices', fontsize=16)
        ax.set_xlabel('Time', fontsize=12)
        ax.set_ylabel('Price ($)', fontsize=12)
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        
    except Exception as e:
        print(f"Error creating aligned dataset: {e}")
        import traceback
        traceback.print_exc()
else:
    print("No multi-ticker data available to create aligned dataset.")

In [ ]:
aligned_data_corr = (
    aligned_data
    .groupby(aligned_data.index.date)            # per day
    .apply(lambda x: x.reindex(
        pd.date_range(
            x.index.min().normalize() + pd.Timedelta("9h30m"),
            x.index.min().normalize() + pd.Timedelta("16h"),
            freq="1min"
        )
    ))
    .droplevel(0)
    .ffill()
)


In [ ]:
aligned_data_corr.to_pickle('full_data_1min_corrected.pkl')